In [1]:
import pandas as pd
import os
import requests
from bs4 import BeautifulSoup
import re
import time
import os, shutil
from pypdf import PdfWriter
import random

In [2]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from pypdf import PdfMerger

In [3]:
def list_include_folder(folder):
    return [os.path.join(dp, f) for dp, dn, fn in os.walk(os.path.expanduser(folder)) for f in fn if 'DS_Store' not in f]

In [4]:
# need to update this to handle the gdrive files
# https://www.adams.edu/administration/institutional-effectiveness/common-data-set/
def extract_data_files(unitid, url, force_driver=False):
    print(f'Scanning {url}')
    domain = extract_domain(url)

    file_extension = url.split('.')[-1]
    if file_extension == 'pdf':
        data_files = pd.DataFrame({
            'link': [url],
            'label': f'{unitid}_base-file',
            'domain': domain,
            'unitid': unitid
        })
        return data_files

    try:
        response = requests.get(url, headers=headers)
    except:
        print(f'WARNING: failed to get {url} with no status code')
        return None

    time.sleep(3)
    if response.status_code == 200 and not force_driver:
        soup = BeautifulSoup(response.content)

        anchors = soup.select('a')
        anchors = [anchor for anchor in anchors if anchor.has_attr('href')]

        pdf_anchors = [anchor for anchor in anchors if 'pdf' in anchor['href']]
        file_anchors = [anchor for anchor in anchors if 'file' in anchor['href'] or 'download' in anchor['href']]
        excel_anchors = [anchor for anchor in anchors if 'xlsx' in anchor['href'] or 'excel' in anchor['href']]
        anchors = pd.Series(pdf_anchors + file_anchors + excel_anchors).unique()

        pdf_links = [anchor['href'] for anchor in anchors]
        anchor_labels = [anchor.text.replace(' ', '') for anchor in anchors]
    elif response.status_code == 403 or force_driver:
        try:
            driver.get(url)
        except:
            return None
        time.sleep(3)
        # driver.execute_script("window.scrollTo(0, 1000)")
        # time.sleep(1)
        anchors = driver.find_elements(By.CSS_SELECTOR, 'a')

        anchors = [anchor for anchor in anchors if anchor.get_attribute('href') is not None]
        
        pdf_anchors = [anchor for anchor in anchors if 'pdf' in anchor.get_attribute('href')]
        file_anchors = [anchor for anchor in anchors if 'file' in anchor.get_attribute('href') or 'download' in anchor.get_attribute('href') or 'document' in anchor.get_attribute('href')]
        excel_anchors = [anchor for anchor in anchors if 'xlsx' in anchor.get_attribute('href') or 'excel' in anchor.get_attribute('href')]
        anchors = pd.Series(pdf_anchors + file_anchors + excel_anchors).unique()

        pdf_links = [anchor.get_attribute('href') for anchor in anchors]
        anchor_labels = [anchor.text.replace(' ', '') for anchor in anchors]
    else:
        print(f'WARNING: failed to get {url} with status code {response.status_code}')
        return None

    data_files = pd.DataFrame({
        'link': pdf_links,
        'label': anchor_labels,
        'domain': domain,
        'unitid': unitid,
        'homepage': url
    })

    return data_files

In [5]:
def extract_data_files_wrapper(unitid, url, force_driver=False):
    try:
        result = extract_data_files(unitid, url, force_driver)
    except:
        print(f'Extracting data files failed for {url}')
        result = None
    return result

In [6]:
def clean_year(year):
    if year is None:
        return 'no-date'
        
    year = year.replace('_', '-')
    if len(year.split('-')) == 1:
        year_num = int(year.strip())
        year = f'{year_num}-{year_num+1}'
    else:
        first_part = year.split('-')[0]
        second_part = year.split('-')[1]
        decade = first_part[0:2]
        if len(second_part) == 2:
            second_part = decade + second_part

        try:
            first_part = int(first_part)
            second_part = int(second_part)
        except:
            return 'no-date'
        
        if second_part != first_part + 1:
            return 'no-date'

        year = f'{first_part}-{second_part}'
    return year

In [39]:
def download_file(url, outpath, overwrite=False):

    if os.path.exists(outpath) and not overwrite:
        print(f'{outpath} already exists')
        return True

    # create the folder, if it doesn't already exist
    if '/' in outpath:
        out_dir = '/'.join(outpath.split('/')[:-1])
        if not os.path.isdir(out_dir):
            os.mkdir(out_dir)

    if 'drive.google.com' in url:
        download_file_from_google_drive(url, outpath)

    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        with open(outpath, 'wb') as out_file:
            out_file.write(response.content)
        print(f'Succesfully downloaded {outpath}')
        return True
    else:
        hard_mode_success = download_file_hard(url, outpath)
        if not hard_mode_success:
            print(f'WARNING: failed to download file {url}')
            return False

In [8]:
def parse_number(x):
    try:
        x = int(x)
        return x
    except:
        return None

In [9]:
def extract_domain(url):
    return url.replace('https://', '').split('/')[0].replace('/', '')

In [10]:
def extract_year(link):
    link = link.split('/')[-1]
    match = re.search('[0-9]{4}[\-_]{1}[0-9]{2,4}', link)
    if match:
        return match.group(0)
    else:
        single_year = re.search('20[0-9]{2}', link)
        if single_year:
            return single_year.group(0)
        return None

In [11]:
def make_https_url(url):
    if 'https://' not in url:
        url = 'https://' + url
    return url

def url_fix_slashes(url):
    url = url.replace('../', '/')
    url = re.sub('/+', '/', url)
    return url

In [12]:
def download_merged_file(links, outpath, overwrite=False, work_dir='../../docs/binding-dir'):

    if os.path.exists(outpath) and not overwrite:
        print(f'{outpath} already exists')
        return True

    clear_folder(work_dir)

    for link in links:
        local_name = link.split('/')[-1]
        temp_path = f'{work_dir}/{local_name}'
        download_file(link, temp_path)

    pdfs = list_include_folder(work_dir)
    if len(pdfs) == 0:
        print(f'No files available to merge for {outpath}')
        return False

    merger = PdfWriter()

    for pdf in pdfs:
        merger.append(pdf)

    merger.write(outpath)
    merger.close()

    clear_folder(work_dir)
    print(f'Succesfully downloaded {outpath}')
    return True


In [13]:
def download_file_hard(url, outpath):
    driver.get(url)
    time.sleep(5)
    files = os.listdir('./selenium-download/')
    if len(files) < 1:
        print(f'WARNING: no downloaded files found for {url}')
    elif len(files) > 1:
        print(f'WARNING: too many files in selenium cache for {url}')
        return False
    else:
        file = f'./selenium-download/{files[0]}'
        os.rename(file, outpath)
        print(f'Succesfully downloaded {url}')
        return True

In [14]:
def fix_domain(domain):
    if domain[-1] != '/':
        domain += '/'
    return domain

In [15]:
def clear_folder(folder):
    for filename in os.listdir(folder):
        file_path = os.path.join(folder, filename)
        try:
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)
        except Exception as e:
            print('Failed to delete %s. Reason: %s' % (file_path, e))

In [16]:
def is_cds(text):
    text = text.lower()
    text = text.replace(' ', '').replace('-', '').replace('_', '')
    return 'cds' in text or 'commondataset' in text

In [17]:
def url_exists(url):
    try:
        response = requests.get(url)
        return response.status_code == 200
    except:
        return False

In [38]:
import sys
import requests


def download_file_from_google_drive(url, destination):
    file_id = extract_google_id(url)
    URL = "https://docs.google.com/uc?export=download&confirm=1"

    session = requests.Session()

    response = session.get(URL, params={"id": file_id}, stream=True)
    token = get_confirm_token(response)

    if token:
        params = {"id": file_id, "confirm": token}
        response = session.get(URL, params=params, stream=True)

    save_response_content(response, destination)


def get_confirm_token(response):
    for key, value in response.cookies.items():
        if key.startswith("download_warning"):
            return value

    return None


def save_response_content(response, destination):
    CHUNK_SIZE = 32768

    with open(destination, "wb") as f:
        for chunk in response.iter_content(CHUNK_SIZE):
            if chunk:  # filter out keep-alive new chunks
                f.write(chunk)

In [42]:
def extract_google_id(url):
    match_obj = re.search('d/(.*)/view', url)
    if match_obj:
        return match_obj.group(1)
    else:
        return None

In [18]:
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/74.0.3729.169 YaBrowser/19.6.1.153 Yowser/2.5 Safari/537.36'}

In [19]:
driver = webdriver.Firefox()

In [20]:
df = pd.read_csv('../../data/landing-pages_master.csv')

In [21]:
df = df.fillna('')
df = df[df['landing_page'] != '']

In [22]:
df = df.rename({
    'UNITID': 'unitid',
    'landing_page': 'href'
}, axis=1)

In [29]:
df['pdf_path'] = [f'../../docs/master/2024-2025/{unitid}.pdf' for unitid in df['unitid']]
df['pdf_path_2025'] = [f'../../docs/master/2025-2026/{unitid}.pdf' for unitid in df['unitid']]
df['excel_path'] = [f'../../docs/excel/2024-2025/{unitid}.xlsx' for unitid in df['unitid']]

df['pdf_exists'] = df['pdf_path'].apply(lambda x: os.path.exists(x))
df['pdf_2025_exists'] = df['pdf_path_2025'].apply(lambda x: os.path.exists(x))
df['excel_exists'] = df['excel_path'].apply(lambda x: os.path.exists(x))
df['file_exists'] = [pdf or pdf_2025 or excel for pdf, pdf_2025, excel in zip(df['pdf_exists'], df['pdf_2025_exists'], df['excel_exists'])]

In [32]:
still_failing = df[df['file_exists'].apply(lambda x: x is False)].reset_index(drop=True)
still_failing.to_csv('../../data/pgrm-control/failures_6.30.csv', index=False)

In [24]:
df = df[df['file_exists'].apply(lambda x: x is False)].reset_index(drop=True)
print(f'{len(df)} landing pages not yet scraped')

421 landing pages not yet scraped


In [25]:
# df['title'] = df['title'].fillna('')
# df = df[df['title'].apply(lambda title: 'common data set' in title.lower() or 'cds' in title.lower())]

In [26]:
result_frames = []
for i in range(0, len(df['unitid'])):
    unitid = df['unitid'][i]
    url = df['href'][i]

    result = extract_data_files_wrapper(unitid, url, force_driver=True)
    if result is not None:
        print(f'Extracted {len(result)} data files')
    else:
        print('No data files found')
    result_frames.append(result)

    if i % 100 == 0:
        print(f'{i / len(df["unitid"])*100}% complete')
        data_files = pd.concat(result_frames)

        data_files.to_csv('../../data/data-files/temp-backup.csv', index=False)

data_files = pd.concat(result_frames)

Scanning https://www.bakersfieldcollege.edu/about/administration/planning-and-accountability/institutional-effectiveness/data-dashboards.html
Extracted 6 data files
0.0% complete
Scanning https://www.cpp.edu/data/cpp-facts.shtml
Extracted 36 data files
Scanning https://aggiedata.ucdavis.edu/common-data-set
Extracted 2 data files
Scanning https://www.ucdenver.edu/offices/institutional-research-and-effectiveness/data-analysis
Extracted 1 data files
Scanning https://www.ir.colostate.edu/common-data-set/
Extracted 1 data files
Scanning https://www.hccfl.edu/doc/college-factbook
Extracted 10 data files
Scanning https://www.georgiasouthern.edu/offices/institutional-research/facts-and-figures
Extracted 23 data files
Scanning https://oir.uic.edu/common-data-set-3/
Extracted 1 data files
Scanning https://ir.library.illinoisstate.edu/cds/
Extracted 0 data files
Scanning https://irds.indianapolis.iu.edu/data-link/index.html
Extracted 4 data files
Scanning https://www.lsu.edu/data/common-data-set/

In [27]:
driver.close()

In [28]:
print(f'Found {len(data_files)} data files')
data_files.to_csv('../../data/pgrm-control/data-files_6-25_10.24.csv', index=False)

Found 3794 data files


# Downloading found data files

In [22]:
data_files = pd.read_csv('../../data/pgrm-control/data-files_6-25_10.24.csv', keep_default_na=False)
# driver = webdriver.Firefox()
# data_files = extract_data_files(166629, 'https://www.umass.edu/uair/data/common-data-set', force_driver=True)
# driver.close()

In [23]:
data_files['link_year'] = data_files['link'].apply(extract_year)
data_files['label_year'] = data_files['label'].apply(extract_year)
data_files['year'] = [link or label for link, label in zip(data_files['link_year'], data_files['label_year'])]
data_files['year'] = data_files['year'].apply(clean_year)

In [24]:
data_files['link_cds'] = data_files['link'].apply(is_cds)
data_files['label_cds'] = data_files['label'].apply(is_cds)
data_files['is_cds'] = [link or label for link, label in zip(data_files['link_cds'], data_files['label_cds'])]
data_files['probation'] = data_files['is_cds'].replace(True, None).replace(False, 'probation')

In [25]:
data_files['is_pdf'] = data_files['link'].apply(lambda x: 'master' if 'pdf' in x else None)
data_files['is_excel'] = data_files['link'].apply(lambda x: 'excel' if 'xlsx' in x or 'excel' in x else None)
data_files['subdirectory'] = [probation or pdf or excel for probation, pdf, excel in zip(data_files['probation'], data_files['is_pdf'], data_files['is_excel'])]
data_files['subdirectory'] = data_files['subdirectory'].fillna('master')
data_files['extension'] = data_files['subdirectory'].replace('master', 'pdf').replace('excel', 'xlsx').replace('probation', 'pdf')

In [26]:
data_files['outpath'] = [f'../../docs/{subdir}/{year}/{unitid}.{extension}' for subdir, year, unitid, extension in zip(data_files['subdirectory'], data_files['year'], data_files['unitid'], data_files['extension'])]

In [34]:
# data_files = data_files[data_files['is_cds']].drop_duplicates().reset_index(drop=True)

In [27]:
multi_files = data_files[data_files['extension'] == 'pdf']
multi_files = multi_files[data_files['year'] != 'no-date']
if len(multi_files) > 0:
    multi_files = multi_files['outpath'].value_counts(ascending=False).reset_index()
    multi_files = multi_files[multi_files['count'].apply(lambda x: x > 1)]
    multi_files['is_multifile'] = True
    multi_files = multi_files[['outpath', "is_multifile"]]

    data_files = data_files.merge(multi_files, how='left')
else:
    data_files['is_multifile'] = False

data_files['is_multifile'] = data_files['is_multifile'].fillna(False)

<positron-console-cell-27>:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
<positron-console-cell-27>:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


In [28]:
# data_files['link'] = data_files['link'].apply(url_fix_slashes)
data_files['url_exists'] = data_files['link'].apply(lambda x: 'http' in x)

In [29]:
data_files['domain'] = data_files['domain'].apply(fix_domain)
basic_urls = data_files[data_files['url_exists']]

advanced_urls = data_files[data_files['url_exists'].apply(lambda x: x is False)]
advanced_urls['link'] = (advanced_urls['domain'] + advanced_urls['link']).apply(make_https_url)

data_files = pd.concat([basic_urls, advanced_urls])

<positron-console-cell-29>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [38]:
# add base files
# base_files = pd.read_csv('https://docs.google.com/spreadsheets/d/1wLcwxakIUdO1JtNZZ6xHWfpZk_tii-QZvXjImsCA8UE/export?gid=2122431272&format=csv')
# base_files = base_files[base_files['is_landing_page'].apply(lambda x: x == 'base_file')].reset_index(drop=True)

# base_files['year'] = base_files['title'].apply(extract_year).apply(clean_year)
# base_files['outpath'] = [f'../docs/master/{year}/{unitid}.pdf' for year, unitid in zip(base_files['year'], base_files['unitid'])]

# base_files = base_files.rename({
#     'href': 'link'
# }, axis=1)

In [39]:
# base_files = base_files[['link', 'outpath']]
# data_files = data_files[['link', 'outpath']]
# data_files = pd.concat([data_files, base_files])

In [31]:
data_files['is_recent'] = data_files['year'].apply(lambda x: parse_number(x.split('-')[0])).apply(lambda x: x is None or x >= 2024)
# data_files = data_files[data_files['is_recent']].reset_index(drop=True)

In [32]:
data_files['new_file'] = data_files['outpath'].apply(lambda x: not os.path.exists(x))
# data_files['new_file'] = True
print(data_files['new_file'].value_counts())
data_files = data_files[data_files['new_file']].drop_duplicates().reset_index(drop=True)

new_file
True     2977
False     817
Name: count, dtype: int64


In [33]:
print(f'{len(data_files)} new files to download')

2734 new files to download


In [34]:
single_files = data_files[data_files['is_multifile'].apply(lambda x: x is False)].reset_index(drop=True)
multi_files = data_files[data_files['is_multifile'].apply(lambda x: x is True)].reset_index(drop=True)

In [35]:
options = webdriver.FirefoxOptions()

options.page_load_strategy = 'none'

# Set preferences directly on FirefoxOptions
options.set_preference("browser.download.folderList", 2)  # 2 means use a custom download path
options.set_preference("browser.download.dir", f"{os.path.abspath('')}/selenium-download")  # Change default directory
options.set_preference("browser.download.manager.showWhenStarting", False)  # Don't show download manager window
options.set_preference("browser.helperApps.neverAsk.saveToDisk", "application/pdf")  # Auto-download PDFs without prompting

# Tell Firefox to save PDFs to disk instead of opening them in the browser preview
options.set_preference("pdfjs.disabled", True)

driver = webdriver.Firefox(options=options)

In [36]:
failed_links = []
for i in range(0, len(single_files['link'])):
    link = single_files['link'][i]
    outpath = single_files['outpath'][i]

    year = single_files['year'][i]
    if year == 'no-date':
        unique_id = link.replace('https', '').replace('http', '').replace(':', '').replace('/', '_').replace('.', '_')
        outpath = outpath.replace('.pdf', '')
        outpath = outpath + '_' + unique_id
        outpath += '.pdf'

    print(outpath)
    try:
        download_file(link, outpath)
    except:
        failed_links.append({
            'link': link,
            'outpath': outpath
        })
        print(f'Error in download: {link}')
    
    if i % 20 == 0:
        print(f'{i / len(data_files["outpath"]) * 100}% complete')

../../docs/probation/no-date/109819___public_tableau_com_app_profile_bc_office_of_institutional_effectiveness_vizzes.pdf
Succesfully downloaded ../../docs/probation/no-date/109819___public_tableau_com_app_profile_bc_office_of_institutional_effectiveness_vizzes.pdf
0.0% complete
../../docs/probation/no-date/109819___public_tableau_com_app_profile_bc_office_of_institutional_effectiveness_viz_WeeklyEnrollmentReport_FrontPage.pdf
Succesfully downloaded ../../docs/probation/no-date/109819___public_tableau_com_app_profile_bc_office_of_institutional_effectiveness_viz_WeeklyEnrollmentReport_FrontPage.pdf
../../docs/probation/no-date/109819___public_tableau_com_app_profile_bc_office_of_institutional_effectiveness_viz_CompletionandEfficiency_V5_07242025_Instructions.pdf
Succesfully downloaded ../../docs/probation/no-date/109819___public_tableau_com_app_profile_bc_office_of_institutional_effectiveness_viz_CompletionandEfficiency_V5_07242025_Instructions.pdf
../../docs/probation/no-date/109819___p

In [37]:
multi_file_paths = multi_files['outpath'].unique()
for outpath in multi_file_paths:
    links = list(multi_files[multi_files['outpath'] == outpath]['link'].unique())
    try:
        download_merged_file(links, outpath)
    except:
        print(f'{outpath} failed multifile download')

Succesfully downloaded ../../docs/binding-dir/Fall-2019-Faculty-Profiles.pdf


invalid pdf header: b'<!DOC'
EOF marker not found


Succesfully downloaded ../../docs/binding-dir/view?usp=drive_link
../../docs/probation/2019-2020/139931.pdf failed multifile download
Succesfully downloaded ../../docs/binding-dir/Fall-2020-Faculty-Profiles.pdf


invalid pdf header: b'<!DOC'
EOF marker not found


Succesfully downloaded ../../docs/binding-dir/view?usp=drive_link
../../docs/probation/2020-2021/139931.pdf failed multifile download
Succesfully downloaded ../../docs/binding-dir/Fall-2021-Faculty-Profiles.pdf


...

Succesfully downloaded ../../docs/binding-dir/nih-guide-for-developing-a-mop-nccih-ver1-07-17-2015-2.docx
../../docs/probation/2015-2016/487010.pdf failed multifile download
Succesfully downloaded ../../docs/binding-dir/Full-2023-2024-Common-Data-Set.pdf
Succesfully downloaded ../../docs/binding-dir/Common-Data-Set-Definitions-07-2023.pdf
Succesfully downloaded ../../docs/master/2023-2024/145691.pdf
Succesfully downloaded ../../docs/binding-dir/General-Information-07-2023.pdf
Succesfully downloaded ../../docs/binding-dir/Enrollment-And-Persistence-07-2023.pdf
Succesfully downloaded ../../docs/binding-dir/Admissions-07-2023.pdf
Succesfully downloaded ../../docs/binding-dir/Transfers-07-2023.pdf
Succesfully downloaded ../../docs/binding-dir/Academic-Offerings-And-Policies-07-2023.pdf
Succesfully downloaded ../../docs/binding-dir/Student-Life-07-2023.pdf
Succesfully downloaded ../../docs/binding-dir/Financial-Aid-07-2023.pdf
Succesfully downloaded ../../docs/binding-dir/Instructional-Facu

Unknown destination: 2010-11_cdsB.pdf [0.0, 1]
Unknown destination: 2010-11_cdsG.pdf [0.0, 1]
Unknown destination: 2010-11_cdsH.pdf [0.0, 1]
Unknown destination: 2010-11_cdsI.pdf [0.0, 1]


Succesfully downloaded ../../docs/binding-dir/2010-11_cdsj.pdf
Succesfully downloaded ../../docs/master/2010-2011/145691.pdf
Succesfully downloaded ../../docs/binding-dir/2009-10_cds.pdf
Succesfully downloaded ../../docs/binding-dir/2009-10_cdsdef.pdf
Succesfully downloaded ../../docs/binding-dir/2009-10_cds_a.pdf
Succesfully downloaded ../../docs/binding-dir/2009-10_cds_b.pdf
Succesfully downloaded ../../docs/binding-dir/2009-10_cds_c.pdf
Succesfully downloaded ../../docs/binding-dir/2009-10_cds_d.pdf
Succesfully downloaded ../../docs/binding-dir/2009-10_cds_e.pdf
Succesfully downloaded ../../docs/binding-dir/2009-10_cds_f.pdf
Succesfully downloaded ../../docs/binding-dir/2009-10_cds_g.pdf
Succesfully downloaded ../../docs/binding-dir/2009-10_cds_h.pdf
Succesfully downloaded ../../docs/binding-dir/2009-10_cds_i.pdf
Succesfully downloaded ../../docs/binding-dir/2009-10_cds_j.pdf
Succesfully downloaded ../../docs/master/2009-2010/145691.pdf
Succesfully downloaded ../../docs/binding-dir/

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)


Succesfully downloaded ../../docs/binding-dir/Cooley%202019%20JD%20Program%20Outcomes.pdf?hsLang=en
Succesfully downloaded ../../docs/probation/2020-2021/172477.pdf
Succesfully downloaded https://antiochcollege.edu/wp-content/uploads/2020/08/Senior_Survey_Summary_2019.pdf
Succesfully downloaded https://antiochcollege.edu/wp-content/uploads/2020/08/Antioch-First-Destination-Results-2019.pdf
Succesfully downloaded ../../docs/probation/2019-2020/245892.pdf


In [38]:
driver.close()

In [39]:
# special_cases = pd.concat([pd.DataFrame(failure, index=[0]) for failure in failed_links]).reset_index(drop=True)
# # special_cases = data_files
# special_cases.to_csv('../data/pgrm-control/failed-downloads.csv', index=False)

In [40]:
results = pd.DataFrame({
    'file': list_include_folder('../../docs/master')
})
results['year'] = results['file'].apply(lambda file: file.split('/')[-2])
coverage = results['year'].value_counts().reset_index()

In [41]:
coverage

,year,count
0,2024-2025,423
1,2022-2023,417
2,2021-2022,411
3,2020-2021,390
4,2023-2024,366
5,2019-2020,351
6,no-date,339
7,2018-2019,307
8,2025-2026,218
9,2017-2018,171
